# 05 · La SNGP

La SNGP (Liu et al., 2023) es una red neuronal con normalización espectral y una capa de Proceso Gaussiano a la salida. La idea: la red transforma las 200 variables en una representación h(x) que aprende sola, y el GP trabaja sobre esa representación en vez de sobre 20 variables elegidas a mano. La normalización espectral (y los bloques residuales) evitan que la red deforme las distancias, así que lo que está lejos en las variables sigue lejos en h(x), y el GP de arriba le da más varianza (ecuación 4 del informe). Las piezas están en `redes.py`.

Acá entrenamos la red, elegimos λ (cuánto pesa la varianza en el score) y la arquitectura **en validación**, y guardamos sus predicciones.

- **Lee:** `outputs/datos_procesados.npz`, `outputs/ids_*.csv`, `outputs/preds/xgboost_*.csv`.
- **Guarda:** `outputs/preds/sngp_vars_calib.csv` y `outputs/preds/sngp_vars_test.csv` (y `sngp2_vars_*`, la ganadora de la grilla, que resultó ser la misma red).
- **En el informe:** el λ = 0 elegido en validación (sección 2.2), la arquitectura elegida en validación (sección 2.3) y la "SNGP" de la Tabla 1, la Tabla 2 y las figuras 2 y 3.

In [5]:
# Bloque 0. Le decimos a torch que use todos los núcleos de la compu, porque entrenar la red en CPU es lo más lento
# del proyecto.
import math, os
import numpy as np
import pandas as pd
import torch

torch.set_num_threads(os.cpu_count())   # os.cpu_count() = cuántos núcleos tiene la compu
print("núcleos:", os.cpu_count(), "| hilos torch:", torch.get_num_threads())

núcleos: 12 | hilos torch: 12


In [6]:
# Bloque 1. Cargamos los datos procesados en el 01 y las predicciones de XGBoost, igual que en el 03.
# Recordemos el cambio de roles: el 10% "calib" ahora es VALIDACIÓN (para el early stopping y para elegir cosas) y el
# 30% "test" es el POOL, donde después corre el experimento conformal.
# Ojo: la fila i de X tiene que ser el mismo solicitante que la fila i de los CSV; los assert frenan todo si no es
# así, en vez de seguir con resultados que estarían mal sin avisar.
import numpy as np
import pandas as pd
import torch

SEED = 42
datos = np.load("outputs/datos_procesados.npz", allow_pickle=True)

# Los nombres de las claves salen del Bloque 0
X_tr = datos["X_train"].astype(np.float32)
y_tr = datos["y_train"].astype(np.float32)
X_va = datos["X_calib"].astype(np.float32)   # el 10%: ahora es VALIDACIÓN
y_va = datos["y_calib"].astype(np.float32)
X_po = datos["X_test"].astype(np.float32)    # el 30%: el POOL de calibración + test
y_po = datos["y_test"].astype(np.float32)

ids_va = pd.read_csv("outputs/ids_calib.csv")["SK_ID_CURR"].to_numpy()
ids_po = pd.read_csv("outputs/ids_test.csv")["SK_ID_CURR"].to_numpy()

# Las predicciones de XGBoost, ordenadas igual que las filas de X
xgb_va = pd.read_csv("outputs/preds/xgboost_calib.csv").set_index("SK_ID_CURR").loc[ids_va]
xgb_po = pd.read_csv("outputs/preds/xgboost_test.csv").set_index("SK_ID_CURR").loc[ids_po]

# Chequeo de alineación: si falla, las filas de X y las de los CSV no están en el mismo orden
assert (xgb_va["TARGET"].to_numpy() == y_va).all(), "validación desalineada"
assert (xgb_po["TARGET"].to_numpy() == y_po).all(), "pool desalineado"

thin_va = xgb_va["thin_file"].to_numpy().astype(bool)
thin_po = xgb_po["thin_file"].to_numpy().astype(bool)

print("formas:", X_tr.shape, X_va.shape, X_po.shape)
print("tasa de mora train/val/pool:",
      y_tr.mean().round(4), y_va.mean().round(4), y_po.mean().round(4))
print("thin-file en el pool:", int(thin_po.sum()),
      "| morosos thin-file en el pool:", int(y_po[thin_po].sum()))

# Carga ok: 184.506 / 30.751 / 92.254 filas con 200 variables, la misma mora (8,07%) en las tres partes, y 13.206
# thin-file en el pool (1.337 morosos).

formas: (184506, 200) (30751, 200) (92254, 200)
tasa de mora train/val/pool: 0.0807 0.0807 0.0807
thin-file en el pool: 13206 | morosos thin-file en el pool: 1337


In [7]:
# Bloque 2. armar_loaders, la misma función del 03: arma los DataLoaders que necesita el entrenamiento para
# cualquier subconjunto de columnas (cols) y de filas de train (idx_train; si es None usa todo train).
# Un DataLoader corta los datos en batches, grupitos de filas que se procesan juntos. Train se mezcla en cada época
# (shuffle) con semilla fija para que sea reproducible; validación no se mezcla porque solo se evalúa.
from torch.utils.data import DataLoader, TensorDataset


def armar_loaders(cols, idx_train=None, batch=512):
    """Loaders de train (mezclado) y validación (sin mezclar), solo con las columnas cols.
    cols: qué columnas (variables) usar. idx_train: qué filas de train (None = todas).
    batch: cuántos solicitantes por tanda en train."""
    Xt = X_tr if idx_train is None else X_tr[idx_train]        # todas las filas o solo las pedidas
    yt = y_tr if idx_train is None else y_tr[idx_train]
    # X[:, cols]: todas las filas, solo las columnas cols; torch.from_numpy pasa de numpy a tensor
    ds_tr = TensorDataset(torch.from_numpy(Xt[:, cols]), torch.from_numpy(yt))
    ds_va = TensorDataset(torch.from_numpy(X_va[:, cols]), torch.from_numpy(y_va))
    gen = torch.Generator().manual_seed(SEED)                  # mezcla reproducible
    tr = DataLoader(ds_tr, batch_size=batch, shuffle=True, generator=gen)   # train: mezclado en cada época
    va = DataLoader(ds_va, batch_size=4096)                    # validación: en orden, tandas grandes
    return tr, va

In [10]:
# Bloque 3. Primera prueba de la SNGP con una submuestra de 30.000 filas de train y las 200 variables, para ver que
# ande antes de gastar tiempo con todo train. Miramos dos cosas en validación: el AUC por segmento (¿se acerca al del
# GP, que daba thin 0.714 y resto 0.758?) y el cociente de las medianas de varianza thin-file / resto (¿da más que 1,
# o sea, la red "duda" más con los thin-file?).
# Los pasos son siempre los mismos cuatro (están en redes.py): armar la red, entrenar con early stopping, una pasada
# por train para armar la matriz P (ajustar_covarianza) y predecir logit y varianza.
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score
from redes import SNGP, entrenar, ajustar_covarianza, predecir

torch.manual_seed(SEED)
rng=np.random.default_rng(SEED)
idx_chico=rng.choice(len(X_tr),size=30000,replace=False) # submuestra de 30.000 sin repetir
cols=np.arange(X_tr.shape[1]) # todas las columnas, 200 variables (0 a 199)

# Armar los loaders: tandas de solicitantes para entrenar (train chico, mezclado)
# y para validar (validación entera). cols = las 200 variables; idx_chico = las 30.000 filas elegidas
tr, va = armar_loaders(cols, idx_train=idx_chico)

# Crear la red: d_in = 200 variables de entrada. El resto va por defecto: 128 neuronas internas, 2 bloques, coef 0.9
red = SNGP(d_in=len(cols))

# Entrenar: lr = tamaño del paso del optimizador (chico = más estable);
# max_epocas = tope de pasadas por los datos; paciencia = cuántas épocas seguidas sin mejorar
# en validación toleramos antes de cortar (early stopping). Devuelve el historial de pérdidas
hist = entrenar(red, tr, va, lr=1e-3, max_epocas=15, paciencia=4)

# Loader para la covarianza: una pasada por los datos de entrenamiento, en tandas grandes de 4096.
# TensorDataset junta variables y etiquetas; torch.from_numpy pasa de array de numpy a tensor de PyTorch
loader_cov = DataLoader(TensorDataset(torch.from_numpy(X_tr[idx_chico]),
                                      torch.from_numpy(y_tr[idx_chico])), batch_size=4096)

# Armar la matriz de covarianza (la "memoria" de por dónde pasaron los datos de entrenamiento).
# Sin este paso no se puede calcular la varianza
ajustar_covarianza(red, loader_cov)

# Predecir en validación: devuelve el logit de default y la varianza de cada solicitante
logit_red, var_red = predecir(red, torch.from_numpy(X_va))

# AUC en tres grupos: todos, solo thin-file (thin_va es un vector de verdadero/falso) y el resto (~ = "no").
# Se usa el logit directo porque el AUC solo mira el orden. round(..., 3) deja 3 decimales
print("AUC total / thin-file / resto:",
      round(roc_auc_score(y_va, logit_red), 3),
      round(roc_auc_score(y_va[thin_va], logit_red[thin_va]), 3),
      round(roc_auc_score(y_va[~thin_va], logit_red[~thin_va]), 3))

# Cociente de la mediana de varianza de thin-file sobre la del resto. Mayor que 1 = la red ve a los thin-file
# como más atípicos; menor que 1 = lo contrario (lo que pasaba con el GP: ~0.79)
print("mediana de varianza thin-file / resto:",
      round(float(np.median(var_red[thin_va]) / np.median(var_red[~thin_va])), 2))

# La pérdida de validación baja de 0.3153 (época 1) a 0.2493 (época 15) y se aplana desde la época ~10, mientras la de
# train sigue bajando: un sobreajuste leve, que con más datos debería mejorar.
# AUC total / thin / resto: 0.750 / 0.711 / 0.755. Con 1/6 de los datos la red ya iguala al GP y le faltan ~0.01 para
# XGBoost (0.762 / 0.725 / 0.767). Cociente de varianzas thin / resto: 1.10 (con el GP daba 0.79). Una primera señal
# a favor de la hipótesis, pero chica y con una submuestra.

época  1 | pérdida train 0.4462 | pérdida val 0.3153
época  2 | pérdida train 0.2903 | pérdida val 0.2777
época  3 | pérdida train 0.2699 | pérdida val 0.2645
época  4 | pérdida train 0.2603 | pérdida val 0.2570
época  5 | pérdida train 0.2554 | pérdida val 0.2540
época  6 | pérdida train 0.2527 | pérdida val 0.2521
época  7 | pérdida train 0.2510 | pérdida val 0.2512
época  8 | pérdida train 0.2502 | pérdida val 0.2515
época  9 | pérdida train 0.2489 | pérdida val 0.2507
época 10 | pérdida train 0.2478 | pérdida val 0.2496
época 11 | pérdida train 0.2473 | pérdida val 0.2496
época 12 | pérdida train 0.2460 | pérdida val 0.2506
época 13 | pérdida train 0.2451 | pérdida val 0.2493
época 14 | pérdida train 0.2437 | pérdida val 0.2495
época 15 | pérdida train 0.2425 | pérdida val 0.2493
AUC total / thin-file / resto: 0.75 0.711 0.755
mediana de varianza thin-file / resto: 1.1


In [11]:
# Bloque 4. Ahora sí, la red con TODO train: los mismos pasos que en el bloque anterior, con más épocas y más
# paciencia porque hay 6 veces más datos. Esta es la SNGP que usamos en todo el trabajo.
torch.manual_seed(SEED)                  # fija el azar: pesos iniciales y mezcla, para poder repetir

# Loaders con TODO train (si no se pasa idx_train, usa todas las filas) y las 200 variables
tr, va = armar_loaders(cols)

# Red nueva: 200 variables de entrada, 128 neuronas internas, 2 bloques, coef 0.9 (valores por defecto)
red = SNGP(d_in=len(cols))

# Entrenar: lr = tamaño del paso; max_epocas = tope de pasadas (ahora 40);
# paciencia = épocas seguidas sin mejorar en validación antes de cortar (ahora 5)
hist = entrenar(red, tr, va, lr=1e-3, max_epocas=40, paciencia=5)

# Loader de TODO train para la covarianza: tandas grandes (4096), sin mezclar.
# TensorDataset junta variables y etiquetas; torch.from_numpy pasa de numpy a tensor
loader_cov = DataLoader(TensorDataset(torch.from_numpy(X_tr),
                                      torch.from_numpy(y_tr)), batch_size=4096)

# Una pasada por train para armar la matriz de covarianza (la "memoria" de dónde hubo datos)
ajustar_covarianza(red, loader_cov)

# Logit de default y varianza de cada solicitante de validación
logit_red, var_red = predecir(red, torch.from_numpy(X_va))

# AUC total / thin-file / resto (el logit sirve directo: el AUC solo mira el orden)
print("AUC total / thin-file / resto:",
      round(roc_auc_score(y_va, logit_red), 3),
      round(roc_auc_score(y_va[thin_va], logit_red[thin_va]), 3),
      round(roc_auc_score(y_va[~thin_va], logit_red[~thin_va]), 3))

# Cociente de medianas de varianza: > 1 = la red ve a los thin-file como más atípicos que el resto
print("mediana de varianza thin-file / resto:",
      round(float(np.median(var_red[thin_va]) / np.median(var_red[~thin_va])), 2))

# El early stopping se queda con la época 9 (pérdida de validación 0.2466). AUC total / thin / resto: 0.756 / 0.722 /
# 0.761. Con todas las variables la red ordena a los thin-file casi como XGBoost en validación (0.722 contra 0.725):
# ya no tiene el problema de información que tenía el GP con 20 variables.
# La varianza mediana de los thin-file es ~12% mayor que la del resto (cociente 1.12); con el GP era al revés (0.79).
# Es la señal que esperábamos, pero chica: falta ver si alcanza para cambiar a quién se aprueba, y eso se mide en el 04.

época  1 | pérdida train 0.2952 | pérdida val 0.2518
época  2 | pérdida train 0.2512 | pérdida val 0.2491
época  3 | pérdida train 0.2497 | pérdida val 0.2483
época  4 | pérdida train 0.2488 | pérdida val 0.2484
época  5 | pérdida train 0.2480 | pérdida val 0.2473
época  6 | pérdida train 0.2472 | pérdida val 0.2469
época  7 | pérdida train 0.2460 | pérdida val 0.2474
época  8 | pérdida train 0.2451 | pérdida val 0.2476
época  9 | pérdida train 0.2444 | pérdida val 0.2466
época 10 | pérdida train 0.2434 | pérdida val 0.2481
época 11 | pérdida train 0.2427 | pérdida val 0.2472
época 12 | pérdida train 0.2419 | pérdida val 0.2482
época 13 | pérdida train 0.2407 | pérdida val 0.2478
época 14 | pérdida train 0.2399 | pérdida val 0.2490
AUC total / thin-file / resto: 0.756 0.722 0.761
mediana de varianza thin-file / resto: 1.12


In [12]:
# Bloque 5. Con la red ya entrenada, calculamos logit y varianza de cada solicitante de validación y del pool y los
# guardamos en CSV, con el mismo formato que gp_vars_*.csv. El 04 los lee de ahí: la regla del proyecto es entrenar
# una vez, guardar predicciones y experimentar sobre las predicciones.
logit_va, var_va = predecir(red, torch.from_numpy(X_va))     # validación (ya lo teníamos como *_red)
logit_po, var_po = predecir(red, torch.from_numpy(X_po))     # pool: el 30% donde se hace el experimento

# Tabla con el ID de cada solicitante, su logit de default y su varianza
pd.DataFrame({"SK_ID_CURR": ids_va, "logit": logit_va, "var": var_va}).to_csv(
    "outputs/preds/sngp_vars_calib.csv", index=False)
pd.DataFrame({"SK_ID_CURR": ids_po, "logit": logit_po, "var": var_po}).to_csv(
    "outputs/preds/sngp_vars_test.csv", index=False)

print("guardado. pool:", len(logit_po), "| varianza mediana thin / resto:",
      round(float(np.median(var_po[thin_po])), 5), "/", round(float(np.median(var_po[~thin_po])), 5))

# Guardados outputs/preds/sngp_vars_calib.csv y sngp_vars_test.csv, 92.254 filas en el pool. Varianza mediana
# thin / resto en el pool: 0.00415 / 0.00369 (cociente 1.12, igual que en validación).
# La varianza de la red (~0.004) es muy chica: con el ajuste mean-field estándar (logit / raíz(1 + pi/8 var)) el
# divisor da ~1.0008 y no cambia el orden de nadie. Para que la varianza pese hay que multiplicarla por un factor
# lambda grande, que elegimos en el bloque que sigue.

guardado. pool: 92254 | varianza mediana thin / resto: 0.00415 / 0.00369


In [13]:
# Bloque 6. Elegimos lambda (en el código, lam) en validación, sin mirar el pool.
# El ajuste por varianza divide el logit por raíz(1 + lam * pi/8 * var). lam = 0 es "sin ajuste" (el logit de la red
# tal cual). Con lam grande, a los que tienen varianza alta se les achica el logit hacia 0 (hacia "50% de mora"): si
# eran seguros, pasan a parecer más riesgosos y quedan más atrás en la fila. Es la forma de "usar" la incertidumbre
# en el score.
# Probamos varios lam y medimos en validación cuántos thin-file aprueba el conformal. El criterio lo fijamos antes de
# ver nada: gana el lam con mayor tasa de aprobación thin-file promedio entre alfa 3% y 9%.
from sklearn.model_selection import train_test_split   # para partir validación en dos mitades al azar
from conformal import conformal_select                 # nuestro método: devuelve quién queda aprobado

# Factores que multiplican la varianza. 0 = red sin ajuste (punto de comparación).
# Van de a saltos grandes porque la varianza es chica (~0.004) y necesita escala.
LAMS = [0, 30, 100, 300, 1000, 3000]

# Niveles de FDR donde evaluamos: de más estricto (3%) a más permisivo (9%).
ALFAS_VAL = [0.03, 0.05, 0.07, 0.09]

# Cuántas veces repetimos la partición aleatoria de validación.
# Más repeticiones = promedio más estable (cada partición sola es ruidosa).
R_VAL = 20


def score(logit, var=None, lam=0.0):
    """Devuelve mu = probabilidad de BUEN pago (alto = solicitante más seguro).
    logit: logit de morosidad que predice la red (alto = más riesgo).
    var:   varianza de esa predicción (incertidumbre).
    lam:   cuánto pesa la varianza; con lam = 0 se ignora."""
    # Si lam > 0, achicamos el logit en proporción a la varianza (ajuste mean-field).
    # Si lam = 0, usamos el logit sin tocar.
    z = logit / np.sqrt(1 + lam * np.pi / 8 * var) if lam > 0 else logit
    # sigmoide(z) = prob. de morosidad; 1 - eso = prob. de buen pago.
    return 1 - 1 / (1 + np.exp(-z))


y_v = y_va.astype(int)                       # TARGET de validación como enteros 0/1 (1 = moroso)

# Etiqueta que combina segmento y TARGET: 0 = resto pagador, 1 = resto moroso,
# 2 = thin pagador, 3 = thin moroso. Sirve para que cada mitad conserve la misma
# proporción de cada grupo (estratificar).
estrato_v = 2 * thin_va.astype(int) + y_v

filas = []                                   # acá se acumulan los resultados de cada corrida

for r in range(R_VAL):                       # una partición distinta por cada r
    # Partimos los índices de validación en calibración (i_cal) y prueba (i_test).
    # test_size=0.5: mitad y mitad.
    # stratify=estrato_v: misma mezcla de grupos en las dos mitades.
    # random_state=r: cada repetición usa una semilla distinta, pero reproducible.
    i_cal, i_test = train_test_split(np.arange(len(y_v)), test_size=0.5,
                                     stratify=estrato_v, random_state=r)

    # Generador de números aleatorios para el desempate del p-valor conformal.
    rng = np.random.default_rng(r)

    # Calibramos y probamos solo con thin-file (conformal por segmento).
    c = i_cal[thin_va[i_cal]]                # índices de calibración que son thin-file
    t = i_test[thin_va[i_test]]              # índices de prueba que son thin-file

    for lam in LAMS:
        mu = score(logit_va, var_va, lam)    # score de todos los de validación con este lam
        for alfa in ALFAS_VAL:
            # conformal_select(mu_cal, y_cal, mu_test, alfa, rng):
            #   mu[c], y_v[c] : scores y TARGET reales de la calibración (para medir morosos)
            #   mu[t]         : scores de los que queremos decidir si aprobar
            #   alfa          : FDR objetivo
            #   rng           : desempates aleatorios
            # Devuelve un vector True/False: True = aprobado.
            aprob = conformal_select(mu[c], y_v[c], mu[t], alfa, rng)
            # .mean() de un vector True/False = fracción aprobada = tasa de aprobación.
            filas.append({"lam": lam, "alfa": alfa, "tasa": aprob.mean()})

# Promedio de las 20 repeticiones: filas = lam, columnas = alfa.
tabla = pd.DataFrame(filas).groupby(["lam", "alfa"])["tasa"].mean().unstack().round(3)

# Columna extra: promedio entre alfas, que es el criterio de elección.
tabla["promedio"] = tabla.mean(axis=1).round(3)

print(tabla)
print("lam elegido (mayor promedio):", tabla["promedio"].idxmax())

# Aprobación thin-file promedio (alfa 3-9%): lam 0 -> 0.632, 30 -> 0.631, 100 -> 0.626, 300 -> 0.611, 1000 -> 0.568,
# 3000 -> 0.540. Cuanto más pesa la varianza, menos thin-file se aprueban (sobre todo a alfa 3% y 5%). Se elige
# lam = 0: en el 04 la SNGP entra sin ajuste por varianza, y la varianza queda solo como medida de qué tan lejos
# está cada solicitante de los datos (es lo que dice la sección 2.2 del informe).
# Intuición: como los thin-file no están más lejos que el resto (cociente 1.12), penalizar por varianza no separa
# mejor a morosos de buenos pagadores; solo mete ruido en el orden.

alfa   0.03   0.05   0.07   0.09  promedio
lam                                       
0     0.168  0.588  0.818  0.952     0.632
30    0.167  0.588  0.817  0.952     0.631
100   0.153  0.583  0.816  0.952     0.626
300   0.132  0.541  0.817  0.953     0.611
1000  0.098  0.409  0.813  0.951     0.568
3000  0.071  0.351  0.791  0.949     0.540
lam elegido (mayor promedio): 0


In [14]:
# Bloque 7. Grilla corta de la arquitectura, elegida en validación. Probamos 6 versiones de la SNGP (más ancha, más
# profunda, bloques más suaves, aprendizaje más lento) y nos quedamos con la que mejor ordena a los thin-file en
# validación (AUC thin). Como el conformal solo usa el orden de los scores, mejor AUC thin = más thin-file aprobados
# al mismo alfa. El pool no se toca.
# Cada configuración entrena la red entera (con early stopping) y tarda unos minutos.
import time
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, TensorDataset
from redes import SNGP, entrenar, ajustar_covarianza, predecir

cols = np.arange(X_tr.shape[1])      # usamos todas las variables

# Cada configuración: ancho de la capa oculta, cantidad de bloques residuales,
# coef (cuánto aporta cada bloque; menor a 1) y lr (velocidad de aprendizaje).
CONFIGS = [
    {"d_oculta": 128, "n_bloques": 2, "coef": 0.9, "lr": 1e-3},   # la que ya tenemos (referencia)
    {"d_oculta": 256, "n_bloques": 2, "coef": 0.9, "lr": 1e-3},   # más ancha
    {"d_oculta": 256, "n_bloques": 3, "coef": 0.9, "lr": 1e-3},   # más ancha y más profunda
    {"d_oculta": 128, "n_bloques": 2, "coef": 0.5, "lr": 1e-3},   # bloques residuales más suaves
    {"d_oculta": 128, "n_bloques": 2, "coef": 0.9, "lr": 3e-4},   # aprende más lento (más fino)
    {"d_oculta": 256, "n_bloques": 2, "coef": 0.9, "lr": 3e-4},   # ancha y lenta
]

# Loader con todo train para ajustar la covarianza del GP al final de cada entrenamiento.
loader_cov = DataLoader(TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)),
                        batch_size=4096)

resultados = []      # una fila por configuración con sus AUC
redes_ok = []        # guardamos cada red entrenada para reusar la ganadora sin reentrenar

for k, cfg in enumerate(CONFIGS):
    t0 = time.time()                                # para medir cuántos minutos tarda cada configuración
    torch.manual_seed(SEED)                         # misma semilla: comparación justa entre configs
    tr, va = armar_loaders(cols)                    # loaders de train y validación (como antes)

    # Armamos la red con los argumentos de esta fila (sin 'lr', que va en entrenar).
    red_k = SNGP(d_in=len(cols), d_oculta=cfg["d_oculta"],
                 n_bloques=cfg["n_bloques"], coef=cfg["coef"])

    # Entrenamos con early stopping sobre la pérdida de validación.
    # verbose=False: no imprime época por época.
    entrenar(red_k, tr, va, lr=cfg["lr"], max_epocas=40, paciencia=5, verbose=False)

    # El GP necesita su matriz de covarianza (para la varianza) con la red ya entrenada.
    ajustar_covarianza(red_k, loader_cov)

    # Predicciones en validación: logit (riesgo) y var (incertidumbre).
    logit_k, var_k = predecir(red_k, torch.from_numpy(X_va))

    # AUC total, thin-file y resto (thin_va: True si el solicitante es thin-file).
    auc_t = roc_auc_score(y_va, logit_k)
    auc_thin = roc_auc_score(y_va[thin_va], logit_k[thin_va])
    auc_resto = roc_auc_score(y_va[~thin_va], logit_k[~thin_va])

    resultados.append({**cfg, "auc_total": round(auc_t, 4), "auc_thin": round(auc_thin, 4),
                       "auc_resto": round(auc_resto, 4), "min": round((time.time() - t0) / 60, 1)})
    redes_ok.append(red_k)
    print(k, resultados[-1])                        # progreso: una línea por configuración

tabla_cfg = pd.DataFrame(resultados)
print(tabla_cfg.to_string())

# Criterio fijado de antemano: mayor AUC thin en validación.
mejor = int(tabla_cfg["auc_thin"].idxmax())         # idxmax = número de fila con el AUC thin más alto
print("configuración elegida:", mejor, CONFIGS[mejor])
print("referencia XGBoost (AUC thin validación): 0.725")

# Con la red elegida, guardamos predicciones de validación y pool en archivos nuevos
# (no pisan los sngp_vars anteriores).
red = redes_ok[mejor]
logit_va, var_va = predecir(red, torch.from_numpy(X_va))
logit_po, var_po = predecir(red, torch.from_numpy(X_po))
pd.DataFrame({"SK_ID_CURR": ids_va, "logit": logit_va, "var": var_va}).to_csv(
    "outputs/preds/sngp2_vars_calib.csv", index=False)
pd.DataFrame({"SK_ID_CURR": ids_po, "logit": logit_po, "var": var_po}).to_csv(
    "outputs/preds/sngp2_vars_test.csv", index=False)
print("guardado sngp2. varianza mediana thin / resto (pool):",
      round(float(np.median(var_po[thin_po])), 5), "/", round(float(np.median(var_po[~thin_po])), 5))

# Las 6 configuraciones dan AUC thin entre 0.718 y 0.722. El error estándar del AUC thin en validación es ~0.014
# (hay ~435 morosos thin-file en validación), así que son indistinguibles: ni más ancha, ni más profunda, ni con coef
# menor, ni con lr menor mejora el orden de los thin-file. La red ya está en su techo con estas entradas.
# Por el criterio fijado gana la configuración 0 (128 de ancho, 2 bloques, coef 0.9, lr 1e-3), que es la misma red
# del Bloque 4: sngp2 = sngp, así que en el resto del trabajo usamos sngp_vars_*.csv.

0 {'d_oculta': 128, 'n_bloques': 2, 'coef': 0.9, 'lr': 0.001, 'auc_total': 0.7562, 'auc_thin': 0.722, 'auc_resto': 0.7608, 'min': 2.0}
1 {'d_oculta': 256, 'n_bloques': 2, 'coef': 0.9, 'lr': 0.001, 'auc_total': 0.7564, 'auc_thin': 0.7214, 'auc_resto': 0.7612, 'min': 2.4}
2 {'d_oculta': 256, 'n_bloques': 3, 'coef': 0.9, 'lr': 0.001, 'auc_total': 0.7568, 'auc_thin': 0.718, 'auc_resto': 0.7623, 'min': 2.1}
3 {'d_oculta': 128, 'n_bloques': 2, 'coef': 0.5, 'lr': 0.001, 'auc_total': 0.7563, 'auc_thin': 0.7202, 'auc_resto': 0.7614, 'min': 1.9}
4 {'d_oculta': 128, 'n_bloques': 2, 'coef': 0.9, 'lr': 0.0003, 'auc_total': 0.7577, 'auc_thin': 0.7215, 'auc_resto': 0.7625, 'min': 3.0}
5 {'d_oculta': 256, 'n_bloques': 2, 'coef': 0.9, 'lr': 0.0003, 'auc_total': 0.7583, 'auc_thin': 0.7193, 'auc_resto': 0.7636, 'min': 3.7}
   d_oculta  n_bloques  coef      lr  auc_total  auc_thin  auc_resto  min
0       128          2   0.9  0.0010     0.7562    0.7220     0.7608  2.0
1       256          2   0.9  0.0010